# Policy Mirror Descent (Exponentiated Gradient)

## Background

Mirror descent (MD) generalises gradient descent to non-Euclidean geometries
by replacing the squared L2 distance with a **Bregman divergence** D_psi:

```
x_{t+1} = argmin_x { eta * <g_t, x> + D_psi(x, x_t) }
```

For policies `pi(.|s)` living on the **probability simplex**, the natural
Bregman divergence is the **KL divergence** (generated by the negative entropy
psi = sum_a pi(a) log pi(a)).

The closed-form MD update for cost minimisation at state s becomes:

```
pi_{t+1}(a|s) ∝ pi_t(a|s) * exp(-eta * Q_t(s, a))
```

This is a **multiplicative update** on the policy probabilities directly
(not on softmax parameters theta). Actions with high Q-values are
exponentially down-weighted.

When Q_t is estimated via a Monte-Carlo rollout and only the visited action
a_t has a return estimate:

```
log pi_{t+1}(a_t|s) = log pi_t(a_t|s) - eta * G_t  [then renormalise]
```

This is the **exponentiated gradient (EG)** algorithm.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from environment.gridworld import GridWorldEnv
from experiments.run_all import load_config
from utils.plotting import (
    plot_multi_curves, plot_value_heatmap, plot_vstar_heatmap,
    plot_policy_arrows, plot_summary_bar, save_figure,
)
from utils.metrics import signed_error, abs_error, policy_eval_error

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

cfg    = load_config()
N_EPS  = cfg['environment']['n_episodes']
MAX_ST = cfg['environment']['max_steps_per_episode']
GAMMA  = cfg['environment']['gamma']
LR     = cfg['reinforce']['lr_policy']

def flat(obs):
    return int(obs[0]) * 8 + int(obs[1])

## 1. Implementation

Key difference from REINFORCE:
- REINFORCE: stores `theta`; update is **additive** in `theta` space
- Mirror Descent: stores `log_pi`; update is **multiplicative** in `pi` space

Both use the same Monte-Carlo return estimate, but the geometry differs.

In [ ]:
class MirrorDescentAgent:
    """
    Episodic Policy Mirror Descent with KL-divergence Bregman potential.

    Update rule per observed (s_t, a_t, G_t):
        log pi_{new}[s, a_t] -= eta * G_t
        log pi_{new}[s, :]   -= log Z_s        (renormalisation)

    Equivalent to:
        pi_{new}(a|s) ∝ pi_old(a|s) * exp(-eta * G_t * I[a = a_t])

    The state is stored as log-probabilities (not theta) so that the
    multiplicative update is simply an additive shift in log space.
    """

    def __init__(self, rng, gamma: float, lr: float,
                 n_states: int = 64, n_actions: int = 9):
        self._rng = rng
        self.gamma = gamma
        self.lr = lr
        self.n_states = n_states
        self.n_actions = n_actions
        # Start from uniform policy: log pi = -log(n_actions) for all (s, a)
        self._log_pi = np.full((n_states, n_actions), -np.log(n_actions))
        self._buf = []

    def _probs(self, s: int) -> np.ndarray:
        lp = self._log_pi[s, :]
        lp = lp - lp.max()           # shift for numerical stability
        p  = np.exp(lp)
        return p / p.sum()

    def select_action(self, s: int) -> int:
        return int(self._rng.choice(self.n_actions, p=self._probs(s)))

    def reset_episode(self):
        self._buf = []

    def update(self, s: int, a: int, cost: float, s_prime: int, done: bool):
        self._buf.append((s, a, float(cost)))

    def finish_episode(self):
        G = 0.0
        for s, a, cost in reversed(self._buf):
            G = cost + self.gamma * G
            # Multiplicative update: decrease log-prob of taken action by eta*G
            self._log_pi[s, a] -= self.lr * G
            # Renormalise to keep log_pi consistent with a valid distribution
            lp   = self._log_pi[s, :]
            logZ = np.log(np.sum(np.exp(lp - lp.max()))) + lp.max()
            self._log_pi[s, :] -= logZ

    def get_policy(self) -> np.ndarray:
        return np.vstack([self._probs(s) for s in range(self.n_states)])

    def get_value_estimate(self) -> np.ndarray:
        return np.zeros(self.n_states)   # no critic

## 2. REINFORCE (additive) for Reference

REINFORCE updates `theta` (softmax parameters) additively. MD updates
`log pi` multiplicatively. We compare both with the same lr.

In [ ]:
def _softmax(t: np.ndarray) -> np.ndarray:
    e = np.exp(t - t.max()); return e / e.sum()


class REINFORCERef:
    """Vanilla REINFORCE (additive SGD) — for comparison with MD."""

    def __init__(self, rng, gamma, lr, n_states=64, n_actions=9):
        self._rng, self.gamma, self.lr = rng, gamma, lr
        self.n_states, self.n_actions = n_states, n_actions
        self.theta = np.zeros((n_states, n_actions))
        self._buf = []

    def select_action(self, s):
        return int(self._rng.choice(self.n_actions, p=_softmax(self.theta[s, :])))

    def reset_episode(self): self._buf = []
    def update(self, s, a, cost, s_prime, done): self._buf.append((s, a, float(cost)))

    def finish_episode(self):
        G = 0.0
        for s, a, cost in reversed(self._buf):
            G = cost + self.gamma * G
            pi = _softmax(self.theta[s, :])
            g  = pi.copy(); g[a] -= 1.0
            self.theta[s, :] += self.lr * G * g

    def get_policy(self):
        return np.vstack([_softmax(self.theta[s, :]) for s in range(self.n_states)])

    def get_value_estimate(self): return np.zeros(self.n_states)

## 3. Experiments

In [ ]:
def run_one_episode(agent, env, max_steps):
    agent.reset_episode()
    obs = env.reset(); s = flat(obs)
    for _ in range(max_steps):
        a = agent.select_action(s)
        obs, cost, done, _ = env.step(a)
        s_prime = flat(obs)
        agent.update(s, a, cost, s_prime, done)
        s = s_prime
        if done: break
    agent.finish_episode()


def multi_run(factory, n_runs=3, n_eps=N_EPS, max_steps=MAX_ST,
              eval_every=50, base_seed=0) -> dict:
    ckpt_eps   = np.arange(eval_every - 1, n_eps, eval_every)
    signed_arr = np.zeros((n_runs, n_eps))
    abs_arr    = np.zeros((n_runs, n_eps))
    policy_arr = np.zeros((n_runs, len(ckpt_eps)))
    for run in range(n_runs):
        rng_a = np.random.default_rng(base_seed + run * 1000)
        rng_e = np.random.default_rng(base_seed + run * 1000 + 1)
        agent = factory(rng_a); env = GridWorldEnv(rng_e)
        ci = 0
        for ep in range(n_eps):
            run_one_episode(agent, env, max_steps)
            signed_arr[run, ep] = signed_error(agent.get_value_estimate())
            abs_arr[run, ep]    = abs_error(agent.get_value_estimate())
            if ci < len(ckpt_eps) and ep == ckpt_eps[ci]:
                policy_arr[run, ci] = policy_eval_error(agent.get_policy()); ci += 1
        while ci < len(ckpt_eps):
            policy_arr[run, ci] = policy_eval_error(agent.get_policy()); ci += 1
    return {'signed_err': signed_arr, 'abs_err': abs_arr,
            'policy_err': policy_arr, 'checkpoint_eps': ckpt_eps,
            'n_runs': n_runs, 'n_episodes': n_eps}

In [ ]:
N_RUNS = 3

print('Mirror Descent (EG) ...')
md_results = multi_run(
    lambda rng: MirrorDescentAgent(rng, GAMMA, LR), N_RUNS)

print('REINFORCE (SGD reference) ...')
rf_results = multi_run(
    lambda rng: REINFORCERef(rng, GAMMA, LR), N_RUNS)

In [ ]:
cmp = {'Mirror Descent (EG)': md_results, 'REINFORCE (SGD)': rf_results}

fig = plot_multi_curves(cmp, metric='policy_err',
                         title='Policy Eval Error — Mirror Descent vs REINFORCE')
save_figure(fig, '07_policy_err')
plt.show()

## 4. Step-size Sensitivity

EG is sensitive to the step size in a different way from SGD:
a large `eta` causes the policy to collapse onto one action very quickly
(exponential in G_t), which can be destructive or beneficial depending on
whether the collapse is toward a good or bad action.

In [ ]:
lr_sweep = [0.001, 0.005, 0.01, 0.05]
lr_md_res = {}
for lr_val in lr_sweep:
    res = multi_run(
        lambda rng, lr=lr_val: MirrorDescentAgent(rng, GAMMA, lr), n_runs=N_RUNS)
    lr_md_res[f'eta={lr_val}'] = res
    print(f'eta={lr_val:<6}  final_policy_err={res["policy_err"][:, -1].mean():.4f}')

fig = plot_multi_curves(lr_md_res, metric='policy_err',
                         title='Mirror Descent — Step-size Sensitivity')
save_figure(fig, '07_lr_sweep')
plt.show()

In [ ]:
# Extract best eta from the sweep above
best_lr_md = min(
    lr_sweep,
    key=lambda lr: lr_md_res[f'eta={lr}']['policy_err'][:, -1].mean()
)
best_lr_md_err = lr_md_res[f'eta={best_lr_md}']['policy_err'][:, -1].mean()
print(f'Best eta (Mirror Descent): {best_lr_md}  '
      f'(policy_err={best_lr_md_err:.4f})')

# Compare with REINFORCE at the same best lr
rf_at_best = multi_run(
    lambda rng: REINFORCERef(rng, GAMMA, best_lr_md), n_runs=N_RUNS)
print(f'REINFORCE at same lr:        '
      f'(policy_err={rf_at_best["policy_err"][:, -1].mean():.4f})')

## 5. Geometric Intuition: Policy Entropy

MD preserves the simplex structure: the updated policy is always a valid
probability distribution. We can track the policy entropy as training progresses
to see how quickly the policy concentrates.

In [ ]:
INV_ST  = {9, 17, 25, 33, 34, 42, 50, 12, 20, 28, 29, 30, 38, 46, 54}
TERMINAL = 36
VALID = [s for s in range(64) if s not in INV_ST and s != TERMINAL]

def policy_entropy(agent) -> float:
    pi = agent.get_policy()   # (64, 9)
    H  = -np.sum(pi * np.log(pi + 1e-12), axis=1)
    return float(H[VALID].mean())


# Track entropy every 50 episodes for MD and REINFORCE
def entropy_run(AgentClass, lr, n_eps=N_EPS, max_steps=MAX_ST, seed=42):
    agent = AgentClass(np.random.default_rng(seed), GAMMA, lr)
    env   = GridWorldEnv(np.random.default_rng(seed + 1))
    H_curve = []
    for ep in range(n_eps):
        run_one_episode(agent, env, max_steps)
        if (ep + 1) % 50 == 0:
            H_curve.append(policy_entropy(agent))
    return np.array(H_curve)


eps_axis = np.arange(50, N_EPS + 1, 50)
H_md = entropy_run(MirrorDescentAgent, LR)
H_rf = entropy_run(REINFORCERef,       LR)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(eps_axis, H_md, label='Mirror Descent')
ax.plot(eps_axis, H_rf, label='REINFORCE (SGD)', linestyle='--')
ax.axhline(np.log(9), color='gray', linestyle=':', label='Uniform entropy = ln(9)')
ax.set_xlabel('Episode')
ax.set_ylabel('Mean policy entropy H(pi(.|s))')
ax.set_title('Policy Entropy During Training')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_figure(fig, '07_entropy')
plt.show()

## 6. Final Policy

In [ ]:
md_agent = MirrorDescentAgent(np.random.default_rng(42), GAMMA, LR)
env = GridWorldEnv(np.random.default_rng(43))
for ep in range(N_EPS):
    run_one_episode(md_agent, env, MAX_ST)
pi_md = md_agent.get_policy()
print(f'Mirror Descent  policy_err={policy_eval_error(pi_md):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_vstar_heatmap(ax=axes[0])
plot_policy_arrows(pi_md, title='Mirror Descent — greedy policy', ax=axes[1])
fig.tight_layout()
save_figure(fig, '07_final_policy')
plt.show()

In [ ]:
final_pol = {k: v['policy_err'][:, -1].mean() for k, v in cmp.items()}
for k, v in final_pol.items():
    print(f'{k:<25}  policy_err={v:.4f}')
fig = plot_summary_bar(final_pol, title='Final Policy Eval Error')
save_figure(fig, '07_summary_bar')
plt.show()

In [ ]:
print('=' * 50)
print('TUNED HYPERPARAMETERS (Mirror Descent) — update config')
print('=' * 50)
print(f'  mirror_descent.lr (eta) : {best_lr_md}')
print()
print('Full lr sweep results:')
for lr in lr_sweep:
    key = f'eta={lr}'
    err = lr_md_res[key]['policy_err'][:, -1].mean()
    marker = ' <-- best' if lr == best_lr_md else ''
    print(f'  eta={lr:<6}  policy_err={err:.4f}{marker}')

## Discussion

**Mirror descent respects the geometry of the simplex.** REINFORCE updates
`theta` additively and then projects back to valid distributions implicitly via
the softmax. MD updates `pi` multiplicatively and always remains on the simplex
exactly. This has theoretical consequences for convergence bounds.

**The EG update is more aggressive for large G_t.** Because the probability of
a_t decreases exponentially in eta * G_t, a single very costly trajectory can
collapse the policy strongly. This sensitivity can be beneficial (fast learning
when G_t is informative) or harmful (if G_t is noisy).

**In the tabular softmax setting**, REINFORCE and Mirror Descent are closely
related:
- REINFORCE: `theta[s, a] += lr * G_t * (pi(a|s) - I[a=a_t])`
- MD (EG):   `log_pi[s, a] -= lr * G_t * I[a=a_t]` (then renormalise)

The difference is whether the gradient step is taken in `theta`-space
(Euclidean) or `log pi`-space (KL). For softmax parametrisation these
are not the same because the mapping theta -> pi is nonlinear.

**Mirror descent is the theoretical foundation for natural gradient methods**
(notebook 08): the natural gradient can be derived as mirror descent with the
KL divergence in *function space* rather than *parameter space*.